In [6]:
import json
import sys
import os
import time
from tqdm import tqdm
from openai import OpenAI
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
import importlib
import csv
import random
from datasets import load_dataset

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Get the model from environment variable
model = "gpt-4.1-mini-2025-04-14"

# Get temperature and max_tokens from environment variables with defaults
temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.7"))
max_tokens = int(os.getenv("OPENAI_MAX_TOKENS", "1000"))

#set client with key from .env
client = OpenAI(api_key=api_key)

In [7]:
def gpt_generate(messages, model=model, temperature=temperature, max_tokens=max_tokens):

    max_retries = 5
    retries = 0

    while True:
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error: {e}")
            retries += 1
            if retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            wait_time = 2 ** retries
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            continue

In [8]:
system_prompt_string = ""

with open(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Princ_from_reasons\prompts\generation_system_prompt.txt', 'r') as infile:
    system_prompt_string = infile.read()

print(system_prompt_string)

system_prompt = {'role': 'developer', 'content':system_prompt_string}

# Identity

You are a helpful assistant that distills user-provided preference justifications into generally applicable principles.

# Instructions

* Consider presented prompts, pairs of responses, and a preference justifications.
* Distill preference justifications into generally applicable principles that remain pertinant to the prompts and responses shown.
* Unless the justification explicitly describes multiple unique principles, only generate one principle.
* Do not reference specific things discussed in the justifications, instead generate principles that capture the large ideas expressed.
* List principles in a numbered format with each principle on a new line.



In [9]:
ds = load_dataset("nvidia/HelpSteer2", data_dir="preference")['train'] 

print(ds)


data_subset = ds.shuffle(seed=123).select(range(700))




Dataset({
    features: ['split', 'prompt', 'response_1', 'response_2', 'preference_strength', 'preference_statement', 'preference_elaboration', 'three_most_similar_preferences', 'all_preferences_unprocessed'],
    num_rows: 9125
})


In [10]:
generated_principles = []



# data_subset = ds.shuffle(seed=123).select(range(700))

data_subset = ds.shuffle(seed=123).select(range(700))




for preference_annotation in tqdm(data_subset):

    for justification in preference_annotation['all_preferences_unprocessed']:

        prompt = preference_annotation['prompt']

        response_1 = preference_annotation['response_1'] 

        response_2 = preference_annotation['response_2']

        preference_strength = justification['strength']

        preference_statement = justification['justification']

        messages = []
        messages.append(system_prompt)

        generation_prompt = ''
        generation_prompt += 'Consider the following prompt, and the two possible responses. /nPrompt: ' + prompt + '\n' + 'Response 1: ' + response_1 + '\n' + 'Response 2: ' + response_2 + '\n'

        generation_prompt += 'Review the following statement which explains why one response was preferred over another. Identify the 1-3 general points that explain why one statement was preferred over the other. Respond by listing these items directly in an actionable format.'

        generation_prompt += '\nPreference Justification: ' + preference_annotation['all_preferences_unprocessed'][0]['justification'] + '\n'

        generation_prompt += 'Principles:'

        messages.append({'role':'user', 'content':generation_prompt})


        response = gpt_generate(messages)

        generated_principles.append({'prompt':prompt, 'response_1':response_1, 'response_2':response_2, 'preference_justification':preference_annotation['all_preferences_unprocessed'][0]['justification'], 'generated_principles':response})


with open('12_12_gpt_generated_principles.json', 'w') as outfile:
    json.dump(generated_principles, outfile, indent=4)

  0%|          | 0/700 [00:00<?, ?it/s]

100%|██████████| 700/700 [55:37<00:00,  4.77s/it]  


In [11]:
def extract_principles(principles_text):
    principles = []
    for line in principles_text.split('\n'):
        line = line.strip()
        if line:
            # Remove numbering or bullet points
            if line[0].isdigit() or (line[0] in ['-', '*', '•']):
                line = line[1:].strip()
            #remove leading 'Principles:' if present
            if line.lower().startswith('principles:'):
                line = line[len('principles:'):].strip()

            #identify & seperate individual principles contained within brackets {}
            if line.startswith('{') and line.endswith('}'):
                line = line[1:-1].strip()
                sub_principles = [p.strip() for p in line.split(',') if p.strip()]
                #remove any remaining brackets
                for prin in sub_principles:
                    prin = prin.replace('{','').replace('}','')
                    principles.append(prin)


            principles.append(line)
    return principles

In [12]:
#Take the outputs of the pipeline & turn them into a json list of just the principles. 
principles_list = []

with open(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\Princ_from_reasons\12_12_gpt_generated_principles.json') as infile:
    pipeline_outputs = json.load(infile)

for item in pipeline_outputs:
    principles = item.get('generated_principles', '')
    principles = extract_principles(principles)
    for prin in principles:
        if prin != '':
            prin = prin.replace('{','').replace('}','').strip()
            #remove any leading '.' characters
            while prin.startswith('.'):
                prin = prin[1:].strip()
            principles_list.append(prin)


with open('12_12_extracted_principles.json', 'w') as outfile:
    json.dump(principles_list, outfile, indent=4)


In [9]:
#Take the same data we used above and turn it into a csv file in the ICAI format.

data_to_csv = data_subset

null_count = 0

data_out = []

#Add headers
data_out.append(['input', 'text_a', 'text_b', 'preferred_text'])

for item in data_to_csv:
    if(item['preference_strength'] == 0):
        # If preference strength is null, randomly chose between text_a and text_b
        preferred = random.choice(['text_a', 'text_b'])
        data_out.append([item['prompt'], item['response_1'], item['response_2'], preferred])
    elif(item['preference_strength'] > 0):
        data_out.append([item['prompt'], item['response_1'], item['response_2'], 'text_b'])
    elif(item['preference_strength'] < 0):
        data_out.append([item['prompt'], item['response_1'], item['response_2'], 'text_a'])

with open('helpsteer2_preference_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(data_out)

